# Local LLM Laptop Tools — llama.cpp Python example

Python3.11+, llama-cpp-python0.3.36 and requests2.34.2. Install in an isolated environment: `python -m pip install llama-cpp-python==0.3.36 requests==2.34.2`. A compiler/CMake may be needed when a wheel is unavailable; consult the official installation guide. This notebook runs a complete public Qwen2.5-0.5B-Instruct Q4_K_M model on CPU. First run downloads about491MB; subsequent runs hash-check the local file. No hosted API, credentials, cloud inference, GUI app or GPU is required. The other four applications and original Ollama screenshots are resource walkthroughs, not tests executed by this notebook.

In [ ]:
from pathlib import Path
import hashlib,json,os,tempfile
import requests
from llama_cpp import Llama

In [ ]:
revision = '9217f5db79a29953eb74d5343926648285ec7e67'
filename = 'qwen2.5-0.5b-instruct-q4_k_m.gguf'
expected_sha256 = '74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db'
model_path = Path(os.environ.get('TDB_GGUF_PATH', str(Path(tempfile.gettempdir()) / 'tdb-local-llm-models' / filename)))
if not model_path.exists():
    model_path.parent.mkdir(parents=True,exist_ok=True)
    url = f'https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/{revision}/{filename}'
    # Public HTTPS download, no token or account configuration is read.
    with requests.get(url,stream=True,timeout=(10,120)) as response:
        response.raise_for_status()
        with model_path.open('xb') as target:
            for chunk in response.iter_content(1024*1024):target.write(chunk)
digest = hashlib.file_digest(model_path.open('rb'),'sha256').hexdigest()
assert digest == expected_sha256, 'Model hash mismatch; do not load this file.'
assert model_path.stat().st_size == 491400032
assert model_path.open('rb').read(4) == b'GGUF'
print('Official Qwen GGUF verified:',model_path.stat().st_size,'bytes')
print('SHA256:',digest)

In [ ]:
llm = Llama(model_path=str(model_path), n_ctx=512, n_threads=4,
            n_gpu_layers=0, seed=7, logits_all=True)
print('Native model loaded for CPU inference; context:',llm.n_ctx())

In [ ]:
prompt = 'This is a prompt'
max_tokens = 100
temperature = 0.3
top_p = 0.1
echo = True
stop = ['Q', '\n']
model_output = llm(prompt,max_tokens=max_tokens,temperature=temperature,
                   top_p=top_p,echo=echo,stop=stop,seed=7)
final_result = model_output['choices'][0]['text'].strip()
print(final_result)
print('Finish reason:',model_output['choices'][0]['finish_reason'])
print('Completion tokens:',model_output['usage']['completion_tokens'])

In [ ]:
import numpy as np
tokens = llm.tokenize(prompt.encode('utf-8'))
assert model_output['usage']['prompt_tokens'] == len(tokens)
assert 0 < model_output['usage']['completion_tokens'] <= max_tokens
assert model_output['usage']['total_tokens'] == len(tokens) + model_output['usage']['completion_tokens']
assert final_result.startswith(prompt)  # echo=True includes the input, not just new text.
continuation = model_output['choices'][0]['text'][len(prompt):]
assert '\n' not in continuation and 'Q' not in continuation
assert np.isfinite(llm.scores).all()
llm.reset()  # Re-evaluate the full prompt rather than reusing a different-sized cached prefix.
repeated = llm(prompt,max_tokens=max_tokens,temperature=temperature,top_p=top_p,echo=True,stop=stop,seed=7)
assert repeated['choices'][0]['text'] == model_output['choices'][0]['text'], (model_output['choices'][0]['text'],repeated['choices'][0]['text'])
print('Actual token accounting, stop strings, finite logits and fixed-seed repetition passed.')

In [ ]:
chat_output = llm.create_chat_completion(messages=[{'role':'user','content':'What is 2 + 2? Answer with only the number.'}],temperature=0,max_tokens=8,seed=7)
chat_text = chat_output['choices'][0]['message']['content']
assert chat_text.strip() == '4', repr(chat_text)
assert chat_output['usage']['completion_tokens'] > 0
print('Actual local chat response:',chat_text.strip())

In [ ]:
llm.close()
assert hashlib.file_digest(model_path.open('rb'),'sha256').hexdigest() == expected_sha256
print('Native model closed; downloaded weights unchanged.')

## Original source appendix

Every original executable block is retained below as nonexecuting text. The original uses a placeholder model path; the maintained walkthrough supplies a pinned complete model instead. Original screenshots are retained in the article, not represented as newly executed GUI or Ollama results.

### Original block 1

````text
pip install llama-cpp-python
````

### Original block 2

````text
from llama_cpp import Llama
````

### Original block 3

````text
# Instanciate the model
my_aweseome_llama_model = Llama(model_path="./MY_AWESOME_MODEL")


prompt = "This is a prompt"
max_tokens = 100
temperature = 0.3
top_p = 0.1
echo = True
stop = ["Q", "\n"]


# Define the parameters
model_output = my_aweseome_llama_model(
       prompt,
       max_tokens=max_tokens,
       temperature=temperature,
       top_p=top_p,
       echo=echo,
       stop=stop,
   )
final_result = model_output["choices"][0]["text"].strip()
````